# Notebook 01 — Da base bruta à base analítica

**Pesquisa:** Fatores associados à proficiência em Língua Portuguesa de estudantes concluintes do Ensino Médio em escolas públicas de Mato Grosso (SAEB 2023).

**Fonte dos códigos e rótulos:** INEP. Dicionário dos microdados do SAEB 2023, arquivo TS_ALUNO_34EM (3ª/4ª séries do Ensino Médio).

Este notebook parte da extração bruta do SAEB 2023 (Base dos Dados/BigQuery) e produz a **base analítica** usada em todas as análises da dissertação. Todas as decisões ficam registradas aqui, na ordem em que são aplicadas:

1. Carregamento e conferência da base bruta
2. Exclusões sequenciais, com o N de cada etapa
3. Comparação entre estudantes excluídos e retidos
4. Recodificação das variáveis
5. Tratamento das respostas ausentes e da resposta "não sei"
6. Verificações finais e exportação

**Entrada:** `saeb2023_mt_bruta_completa.csv` (ou `.parquet`), com 24.464 registros, extraída pela consulta SQL registrada em `consulta_saeb2023_mt.sql`.

**Saídas:**
- `base_analitica.parquet` e `base_analitica.csv`: base usada em todas as análises
- `grupo_sexo_nao_declarado.parquet`: estudantes que optaram por não declarar o sexo, para a análise complementar
- `tabelas_base_analitica.xlsx`: fluxo de exclusões, comparação excluídos × retidos, dicionário de recodificação e tratamento das ausências
- `registro_base_analitica.json`: versões das bibliotecas, data de execução e números de controle

Nenhuma etapa deste notebook é aleatória; não há semente a registrar.

## 0. Configuração

In [1]:
import sys, json, datetime
import numpy as np
import pandas as pd

# No Google Colab: monta o Drive. Fora do Colab, este bloco é ignorado.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/dissertacao/'   # ajuste para a sua pasta
except ImportError:
    PASTA = './'

ARQ_BRUTA = PASTA + 'saeb2023_mt_bruta_completa.csv'

VERSOES = {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__}
print(VERSOES)

{'python': '3.12.3', 'pandas': '3.0.2', 'numpy': '2.4.4'}


## 1. Carregamento e conferência da base bruta

As respostas do questionário vêm codificadas por letras (A, B, C...). Segundo o dicionário do INEP, **"."** indica resposta em branco e **"\*"** indica resposta nula (marcação dupla). Por isso todas as colunas do questionário são lidas como texto.

A variável `serie` segue o dicionário do INEP: **12 = 3ª/4ª séries do Ensino Médio Tradicional** e **13 = 3ª/4ª séries do Ensino Médio Integrado**.

In [2]:
NUMERICAS = ['ano', 'id_escola', 'id_turma', 'id_aluno', 'serie', 'localizacao', 'area',
             'indicador_proficiencia', 'amostra', 'estrato', 'peso_aluno',
             'proficiencia', 'erro_padrao', 'proficiencia_saeb', 'erro_padrao_saeb',
             'inse', 'nivel_inse', 'peso_inse']

if ARQ_BRUTA.endswith('.parquet'):
    bruta = pd.read_parquet(ARQ_BRUTA)
else:
    bruta = pd.read_csv(ARQ_BRUTA, dtype=str, keep_default_na=False, na_values=[''])
for c in NUMERICAS:
    bruta[c] = pd.to_numeric(bruta[c], errors='coerce')

N_ESPERADO = 24464   # contagem obtida diretamente no BigQuery com os mesmos filtros
assert len(bruta) == N_ESPERADO, f'Base com {len(bruta)} linhas; esperado {N_ESPERADO}.'
assert bruta['id_aluno'].is_unique, 'Há estudantes repetidos na base bruta.'

print('Registros:', len(bruta), '| Escolas:', bruta['id_escola'].nunique(), '| Turmas:', bruta['id_turma'].nunique())
print('Série (12 = EM tradicional, 13 = EM integrado):', bruta['serie'].value_counts().sort_index().to_dict())
print('Amostra (0 = parte censitária):', bruta['amostra'].value_counts().to_dict())

Registros: 24464 | Escolas: 408 | Turmas: 1283
Série (12 = EM tradicional, 13 = EM integrado): {12: 23131, 13: 1333}
Amostra (0 = parte censitária): {0: 24464}


### Escala da proficiência

Segundo o dicionário do INEP, `proficiencia` está na escala única do SAEB com média 0 e desvio 1 **na população de referência** (e não nesta base), e `proficiencia_saeb` é a mesma medida transformada para a escala com média 250 e desvio 50 do SAEB/97. A célula abaixo estima a relação linear entre as duas, que permite converter coeficientes para pontos da escala SAEB.

In [3]:
v = bruta.dropna(subset=['proficiencia', 'proficiencia_saeb'])
b, a = np.polyfit(v['proficiencia'], v['proficiencia_saeb'], 1)
r = np.corrcoef(v['proficiencia'], v['proficiencia_saeb'])[0, 1]
FATOR_SAEB = round(b, 3)
print(f'proficiencia_saeb = {a:.3f} + {b:.3f} × proficiencia   (r = {r:.6f})')

proficiencia_saeb = 249.985 + 55.093 × proficiencia   (r = 1.000000)


## 2. Exclusões sequenciais

O fluxo começa pelos estudantes registrados nos microdados. A contagem dos ausentes na prova foi feita diretamente no BigQuery, com os mesmos filtros da extração, mas sem o filtro de presença: 24.464 presentes e 83 ausentes (participação de 99,7%). Essa taxa refere-se aos estudantes registrados nas turmas avaliadas, e não ao total de matrículas do Censo Escolar.

Na base extraída, as exclusões são aplicadas nesta ordem, e cada estudante é contado apenas na primeira regra que o exclui:

1. estudantes sem proficiência calculada;
2. sexo sem resposta válida (em branco, marcação dupla ou categoria C);
3. raça/cor sem resposta válida (em branco, marcação dupla ou categoria F);
4. pretensão futura em branco ou com marcação dupla.

Pelo dicionário do INEP, a categoria C do sexo e a categoria F da raça/cor correspondem a "Não quero declarar". Essas três variáveis entram como categorias no modelo e não comportam imputação de valor, por isso os casos sem resposta válida são excluídos.

In [4]:
REGRAS = [
    ('Sem proficiência calculada',
        lambda d: (d['indicador_proficiencia'] != 1) | d['proficiencia'].isna()),
    ('Sexo sem resposta válida',
        lambda d: ~d['sexo'].isin(['A', 'B'])),
    ('Raça/cor sem resposta válida',
        lambda d: ~d['raca_cor'].isin(['A', 'B', 'C', 'D', 'E'])),
    ('Pretensão futura sem resposta válida',
        lambda d: ~d['pretensao_futura'].isin(['A', 'B', 'C', 'D'])),
]

# Participação: contagem feita diretamente no BigQuery, sem o filtro de presença
# (SELECT presenca, COUNT(*) ... GROUP BY presenca): 24.464 presentes e 83 ausentes.
N_AUSENTES = 83
N_REGISTRADOS = N_ESPERADO + N_AUSENTES
print(f'Participação na prova de LP: {N_ESPERADO} de {N_REGISTRADOS} registrados ({100 * N_ESPERADO / N_REGISTRADOS:.1f}%)')

atual = bruta.copy()
fluxo = [{'Etapa': 'Estudantes registrados nos microdados (MT, escola pública, LP, 3ª/4ª série, EM tradicional e integrado)',
          'Excluídos': 0, 'Restantes': N_REGISTRADOS},
         {'Etapa': 'Ausentes na prova de Língua Portuguesa', 'Excluídos': N_AUSENTES, 'Restantes': len(atual)}]
excluidos = []
for nome, regra in REGRAS:
    m = regra(atual)
    saida = atual[m].copy()
    saida['motivo_exclusao'] = nome
    excluidos.append(saida)
    atual = atual[~m]
    fluxo.append({'Etapa': nome, 'Excluídos': int(m.sum()), 'Restantes': len(atual)})

fluxo = pd.DataFrame(fluxo)
excluidos = pd.concat(excluidos)
base = atual.copy()
print(fluxo.to_string(index=False))

Participação na prova de LP: 24464 de 24547 registrados (99.7%)
                                                                                                  Etapa  Excluídos  Restantes
Estudantes registrados nos microdados (MT, escola pública, LP, 3ª/4ª série, EM tradicional e integrado)          0      24547
                                                                 Ausentes na prova de Língua Portuguesa         83      24464
                                                                             Sem proficiência calculada         10      24454
                                                                               Sexo sem resposta válida        519      23935
                                                                           Raça/cor sem resposta válida        487      23448
                                                                   Pretensão futura sem resposta válida        452      22996


## 3. Comparação entre excluídos e retidos

Se os estudantes excluídos tiverem proficiência diferente dos retidos, a exclusão altera a média da base. A tabela compara os dois grupos (apenas excluídos com proficiência calculada) e mostra a média da base antes e depois das exclusões.

In [5]:
exc_prof = excluidos[excluidos['motivo_exclusao'] != 'Sem proficiência calculada']

def resumo(nome, d):
    return {'Grupo': nome, 'N': len(d),
            'Proficiência média': round(d['proficiencia'].mean(), 4),
            'Desvio-padrão': round(d['proficiencia'].std(), 4),
            'Média na escala SAEB': round(d['proficiencia_saeb'].mean(), 1)}

comparacao = pd.DataFrame(
    [resumo('Retidos (base analítica)', base)]
    + [resumo(f'Excluídos: {m}', g) for m, g in exc_prof.groupby('motivo_exclusao', sort=False)]
    + [resumo('Todos os excluídos com proficiência', exc_prof),
       resumo('Base antes das exclusões (com proficiência)', pd.concat([base, exc_prof]))])
print(comparacao.to_string(index=False))

                                          Grupo     N  Proficiência média  Desvio-padrão  Média na escala SAEB
                       Retidos (base analítica) 22996              0.3547         0.9134                 269.5
            Excluídos: Sexo sem resposta válida   519              0.1523         0.9738                 258.4
        Excluídos: Raça/cor sem resposta válida   487              0.0018         0.9368                 250.1
Excluídos: Pretensão futura sem resposta válida   452             -0.1648         0.9068                 240.9
            Todos os excluídos com proficiência  1458              0.0037         0.9493                 250.2
    Base antes das exclusões (com proficiência) 24454              0.3338         0.9193                 268.4


## 4. Recodificação

Cada variável é convertida das letras do questionário para valores numéricos segundo o dicionário abaixo. Respostas em branco ou com marcação dupla ficam temporariamente como ausentes (`NaN`) e são tratadas na seção 5. A opção "não sei" da escolaridade dos responsáveis (letra F) é mantida separadamente, pois é uma resposta com significado próprio.

Os enunciados e os rótulos das alternativas foram transcritos do dicionário do INEP. A tabela gerada nesta seção corresponde ao Quadro 3 da dissertação.

In [6]:
TEMPO = {'A': (0, 'Não uso meu tempo para isso'), 'B': (1, 'Menos de 1 hora'), 'C': (2, 'Entre 1 e 2 horas'), 'D': (3, 'Mais de 2 horas')}
NAO_SIM = {'A': (0, 'Não'), 'B': (1, 'Sim')}
FREQ = {'A': (0, 'Nunca ou quase nunca'), 'B': (1, 'De vez em quando'), 'C': (2, 'Sempre ou quase sempre')}
CONCORDA = {'A': (0, 'Concordo totalmente'), 'B': (1, 'Concordo'), 'C': (2, 'Discordo'), 'D': (3, 'Discordo totalmente')}
PROFESSORES = {'A': (0, 'Todos eles'), 'B': (1, 'A maior parte deles'), 'C': (2, 'Poucos deles'), 'D': (3, 'Nenhum deles')}
ESCOL = {'A': (1, 'Não completou o 5º ano do Ensino Fundamental'), 'B': (2, 'Ensino Fundamental, até o 5º ano'),
         'C': (3, 'Ensino Fundamental completo'), 'D': (4, 'Ensino Médio completo'),
         'E': (5, 'Ensino Superior completo')}   # F = "Não sei": tratado na seção 5

DICIONARIO = {
    # variável: (questão INEP, enunciado, {letra: (valor, rótulo)})
    'idade': ('Q02', 'Qual é a sua idade?',
              {'A': (16, '16 anos ou menos'), 'B': (17, '17 anos'), 'C': (18, '18 anos'),
               'D': (19, '19 anos'), 'E': (20, '20 anos'), 'F': (21, '21 anos ou mais')}),
    'reprovacao': ('Q19', 'Você já foi reprovado(a)?',
                   {'A': (0, 'Não'), 'B': (1, 'Sim, uma vez'), 'C': (2, 'Sim, duas vezes ou mais')}),
    'aluno_motivacao_aplicar_aprendizado': ('Q23b', 'Eu me sinto motivado(a), no dia a dia, a usar o que foi ensinado.', CONCORDA),
    'tempo_estudos': ('Q21a', 'Fora da escola em dias de aula, quanto tempo você usa para: estudar (lição de casa, trabalhos escolares etc.)', TEMPO),
    'tempo_cursos': ('Q21b', 'Fora da escola em dias de aula, quanto tempo você usa para: fazer cursos ou atividades extracurriculares', TEMPO),
    'tempo_trabalho_domestico': ('Q21c', 'Fora da escola em dias de aula, quanto tempo você usa para: trabalhar em casa', TEMPO),
    'tempo_trabalho': ('Q21d', 'Fora da escola em dias de aula, quanto tempo você usa para: trabalhar fora de casa', TEMPO),
    'possui_wifi': ('Q13b', 'Na sua casa tem: rede Wi-Fi', NAO_SIM),
    'qtd_computadores': ('Q12b', 'Quantos existem na sua casa: computador (ou notebook)',
                         {'A': (0, 'Nenhum'), 'B': (1, '1'), 'C': (2, '2'), 'D': (3, '3 ou mais')}),
    'possui_escrivaninha': ('Q13d', 'Na sua casa tem: mesa para estudar', NAO_SIM),
    'possui_casa_quarto_individual': ('Q13c', 'Na sua casa tem: um quarto só seu', NAO_SIM),
    'mora_mae': ('Q07a', 'Normalmente, quem mora na sua casa: mãe(s) ou madrasta(s)', NAO_SIM),
    'mora_pai': ('Q07b', 'Normalmente, quem mora na sua casa: pai(s) ou padrasto(s)', NAO_SIM),
    'escolaridade_mae': ('Q08', 'Qual é a maior escolaridade da sua mãe (ou madrasta ou mulher responsável por você)?', ESCOL),
    'escolaridade_pai': ('Q09', 'Qual é a maior escolaridade de seu pai (ou padrasto ou homem responsável por você)?', ESCOL),
    'responsaveis_incentivam_estudo': ('Q10c', 'Com que frequência seus pais ou responsáveis costumam: incentivar você a estudar', FREQ),
    'responsaveis_conversam_escola': ('Q10b', 'Com que frequência seus pais ou responsáveis costumam: conversar com você sobre o que acontece na escola', FREQ),
}
# Itens das questões 22 e 23, usados na verificação de aquiescência (sem imputação)
BATERIA_Q22 = ['professor_informa_conteudo_inicio_ano', 'professor_investiga_conhecimento_previo',
               'professor_contextualiza_conteudo_cotidiano', 'professor_aborda_desigualdade_racial',
               'professor_aborda_desigualdade_genero', 'professor_aborda_bullying_violencia',
               'professor_desenvolve_trabalho_grupo', 'professor_aborda_futuro_profissional']   # escala PROFESSORES
BATERIA_Q23 = ['aluno_interesse_conteudos', 'sala_espaco_diferentes_opinioes',
               'aluno_sente_seguranca_escola', 'aluno_conforto_discordar_professor',
               'aluno_capacidade_argumentacao', 'avaliacao_representa_aprendizado',
               'professor_acredita_capacidade_aluno', 'professor_motiva_continuidade_estudos']   # escala CONCORDA

def recodificar(df):
    """Converte as respostas do questionário (letras) nos valores numéricos do dicionário.
    Branco, nulo e "não sei" ficam como NaN; não há preenchimento nesta etapa."""
    df = df.assign(qtd_computadores=df['possui_computador'])   # Q12b: quantidade de computadores
    r = pd.DataFrame(index=df.index)
    for c in ['id_aluno', 'id_escola', 'id_turma', 'serie', 'estrato', 'peso_aluno',
              'proficiencia', 'erro_padrao', 'proficiencia_saeb', 'erro_padrao_saeb', 'inse', 'nivel_inse']:
        r[c] = df[c]
    r['em_integrado'] = (df['serie'] == 13).astype(int)
    r['localizacao_rural'] = (df['localizacao'] == 2).astype(int)
    r['sexo'] = df['sexo'].map({'A': 'Masculino', 'B': 'Feminino', 'C': 'Não declarado'})
    r['raca_cor'] = df['raca_cor'].map({'A': 'Branca', 'B': 'Preta', 'C': 'Parda', 'D': 'Amarela', 'E': 'Indígena'})
    r['pretensao_futura'] = df['pretensao_futura'].map({'A': 'Somente estudar', 'B': 'Somente trabalhar',
                                                        'C': 'Estudar e trabalhar', 'D': 'Ainda não sabe'})
    for v, (_, _, mapa) in DICIONARIO.items():
        r[v] = df[v].map({k: val for k, (val, _) in mapa.items()})
        r['ausente_' + v] = df[v].isin(['.', '*']).astype(int)
    for v in ['escolaridade_mae', 'escolaridade_pai']:
        r['nao_sabe_' + v] = (df[v] == 'F').astype(int)
    for v in BATERIA_Q22:
        r['q22_' + v] = df[v].map({k: val for k, (val, _) in PROFESSORES.items()})
    for v in BATERIA_Q23:
        r['q23_' + v] = df[v].map({k: val for k, (val, _) in CONCORDA.items()})
    return r

base = base.assign(qtd_computadores=base['possui_computador'])
an = recodificar(base)

# Estudantes que optaram por não declarar o sexo (categoria C), com proficiência e com raça/cor e
# pretensão futura válidas. Não entram na base analítica; são guardados para a análise complementar.
sel = bruta[(bruta['indicador_proficiencia'] == 1) & bruta['proficiencia'].notna()
            & (bruta['sexo'] == 'C')
            & bruta['raca_cor'].isin(['A', 'B', 'C', 'D', 'E'])
            & bruta['pretensao_futura'].isin(['A', 'B', 'C', 'D'])]
nao_declarado = recodificar(sel)
print('Estudantes que não declararam o sexo (grupo complementar):', len(nao_declarado))

dic_tab = pd.DataFrame([{'Variável': v, 'Questão (INEP)': q, 'Enunciado': e,
                         'Codificação': '; '.join(f'{k}→{val} = {rot}' for k, (val, rot) in m.items())}
                        for v, (q, e, m) in DICIONARIO.items()])
dic_tab.loc[dic_tab['Variável'].str.startswith('escolaridade'), 'Codificação'] += '; F = Não sei (ver seção 5)'
print(dic_tab[['Variável', 'Questão (INEP)', 'Codificação']].to_string(index=False))

Estudantes que não declararam o sexo (grupo complementar): 202
                           Variável Questão (INEP)                                                                                                                                                                                                           Codificação
                              idade            Q02                                                                                                       A→16 = 16 anos ou menos; B→17 = 17 anos; C→18 = 18 anos; D→19 = 19 anos; E→20 = 20 anos; F→21 = 21 anos ou mais
                         reprovacao            Q19                                                                                                                                                          A→0 = Não; B→1 = Sim, uma vez; C→2 = Sim, duas vezes ou mais
aluno_motivacao_aplicar_aprendizado           Q23b                                                                                            

## 5. Tratamento das respostas ausentes e da resposta "não sei"

Regras adotadas:

- **Escolaridade da mãe e do pai.** A resposta "não sei" é tratada como informação: recebe um indicador próprio (`nao_sabe_escolaridade_*`). Respostas em branco ou com marcação dupla recebem outro indicador (`ausente_escolaridade_*`). Nos dois casos, a variável ordinal recebe a categoria mais frequente entre as respostas válidas, apenas para que o estudante permaneça no modelo; a diferença desses grupos é estimada pelos indicadores.
- **Item Q23b, moradia com a mãe e moradia com o pai.** Por sua relevância para a interpretação, cada uma recebe um indicador de resposta ausente (`ausente_*`), e o valor ausente é preenchido com a categoria mais frequente.
- **Demais variáveis do questionário.** O valor ausente é preenchido com a categoria mais frequente, e a variável `n_itens_ausentes` registra quantos desses itens o estudante deixou sem resposta válida.
- **Itens das questões 22 e 23** usados apenas na verificação de aquiescência não recebem preenchimento.

Para todas as variáveis, a base guarda o indicador `ausente_<variável>` (e `nao_sabe_<variável>` para a escolaridade). Isso permite que as estatísticas descritivas sejam calculadas apenas com respostas válidas, enquanto os modelos usam a base completa.

A tabela gerada abaixo documenta, para cada variável, o número de respostas em branco, de marcações duplas e de "não sei", a regra aplicada e o valor atribuído.

In [7]:
CHAVE = ['aluno_motivacao_aplicar_aprendizado', 'mora_mae', 'mora_pai']
ESCOLARIDADE = ['escolaridade_mae', 'escolaridade_pai']
DEMAIS = [v for v in DICIONARIO if v not in CHAVE + ESCOLARIDADE]

linhas = []
for v in DICIONARIO:
    bruto = base[v]
    branco, dupla = int((bruto == '.').sum()), int((bruto == '*').sum())
    nao_sabe = int((bruto == 'F').sum()) if v in ESCOLARIDADE else 0
    moda = an[v].mode().iloc[0]            # moda calculada apenas sobre respostas válidas

    if v in ESCOLARIDADE:
        regra = 'Indicadores de "não sei" e de ausência no modelo; ordinal preenchida com a moda'
    elif v in CHAVE:
        regra = 'Indicador de ausência no modelo; valor preenchido com a moda'
    else:
        regra = 'Valor preenchido com a moda; ausência contada em n_itens_ausentes'

    an[v] = an[v].fillna(moda)
    linhas.append({'Variável': v, 'Em branco': branco, 'Marcação dupla': dupla, 'Não sei': nao_sabe,
                   'Total sem resposta válida': branco + dupla + nao_sabe,
                   '% da base': round(100 * (branco + dupla + nao_sabe) / len(base), 2),
                   'Regra': regra, 'Valor atribuído': moda, 'N no modelo': len(an)})

an['n_itens_ausentes'] = base[DEMAIS].isin(['.', '*']).sum(axis=1)
tratamento = pd.DataFrame(linhas)
print(tratamento.to_string(index=False))
print('\nDistribuição de n_itens_ausentes:', an['n_itens_ausentes'].value_counts().sort_index().to_dict())

                           Variável  Em branco  Marcação dupla  Não sei  Total sem resposta válida  % da base                                                                           Regra  Valor atribuído  N no modelo
                              idade         34               4        0                         38       0.17               Valor preenchido com a moda; ausência contada em n_itens_ausentes             17.0        22996
                         reprovacao         96               6        0                        102       0.44               Valor preenchido com a moda; ausência contada em n_itens_ausentes              0.0        22996
aluno_motivacao_aplicar_aprendizado        174              27        0                        201       0.87                    Indicador de ausência no modelo; valor preenchido com a moda              1.0        22996
                      tempo_estudos        107              37        0                        144       0.63           

## 6. Verificações finais e exportação

In [8]:
MODELO = (['sexo', 'raca_cor', 'pretensao_futura'] + list(DICIONARIO)
          + ['nao_sabe_escolaridade_mae', 'nao_sabe_escolaridade_pai']
          + ['ausente_' + v for v in CHAVE + ESCOLARIDADE] + ['n_itens_ausentes'])

assert an[MODELO].isna().sum().sum() == 0, 'Há valores ausentes nas variáveis do modelo.'
assert an['id_aluno'].is_unique
assert len(an) == fluxo['Restantes'].iloc[-1]
assert len(an) + len(excluidos) == N_ESPERADO

controle = {
    'data_execucao': datetime.datetime.now().isoformat(timespec='seconds'),
    'versoes': VERSOES,
    'n_registrados_microdados': N_REGISTRADOS,
    'n_ausentes_prova_lp': N_AUSENTES,
    'n_base_bruta': N_ESPERADO,
    'n_base_analitica': len(an),
    'n_escolas': int(an['id_escola'].nunique()),
    'n_turmas': int(an['id_turma'].nunique()),
    'n_em_integrado': int(an['em_integrado'].sum()),
    'n_sexo_nao_declarado_complementar': len(nao_declarado),
    'proficiencia_media': round(an['proficiencia'].mean(), 4),
    'proficiencia_dp': round(an['proficiencia'].std(), 4),
    'proficiencia_saeb_media': round(an['proficiencia_saeb'].mean(), 2),
    'fator_conversao_saeb': FATOR_SAEB,
}
print(json.dumps(controle, indent=2, ensure_ascii=False))

an.to_parquet(PASTA + 'base_analitica.parquet', index=False)
an.to_csv(PASTA + 'base_analitica.csv', index=False)
nao_declarado.to_parquet(PASTA + 'grupo_sexo_nao_declarado.parquet', index=False)
with pd.ExcelWriter(PASTA + 'tabelas_base_analitica.xlsx') as xw:
    fluxo.to_excel(xw, sheet_name='Fluxo de exclusões', index=False)
    comparacao.to_excel(xw, sheet_name='Excluídos x retidos', index=False)
    dic_tab.to_excel(xw, sheet_name='Dicionário', index=False)
    tratamento.to_excel(xw, sheet_name='Tratamento das ausências', index=False)
with open(PASTA + 'registro_base_analitica.json', 'w', encoding='utf-8') as f:
    json.dump(controle, f, indent=2, ensure_ascii=False)
print('Arquivos salvos em', PASTA)

{
  "data_execucao": "2026-09-25T18:17:54",
  "versoes": {
    "python": "3.12.3",
    "pandas": "3.0.2",
    "numpy": "2.4.4"
  },
  "n_registrados_microdados": 24547,
  "n_ausentes_prova_lp": 83,
  "n_base_bruta": 24464,
  "n_base_analitica": 22996,
  "n_escolas": 408,
  "n_turmas": 1283,
  "n_em_integrado": 1285,
  "n_sexo_nao_declarado_complementar": 202,
  "proficiencia_media": 0.3547,
  "proficiencia_dp": 0.9134,
  "proficiencia_saeb_media": 269.53,
  "fator_conversao_saeb": 55.093
}


Arquivos salvos em ./
